<a href="https://colab.research.google.com/github/fboldt/aulas-am-bsi/blob/main/aula19a_sele%C3%A7%C3%A3o_de_caracter%C3%ADsticas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [64]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

data = fetch_openml(name='arrhythmia', version=1)
X, y = data.data, data.target
X = SimpleImputer(strategy='median').fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [65]:
from sklearn.linear_model import RidgeClassifier
model = RidgeClassifier()

def evaluate(model, data):
  X_train, X_test, y_train, y_test = data
  model.fit(X_train, y_train)
  return model.score(X_test, y_test)

data_original = (X_train, X_test, y_train, y_test)

evaluate(model, data_original)

0.5824175824175825

Redução de dimensionalidade por extração de características a partir do domínio das características

In [66]:
from sklearn.decomposition import PCA
pca = PCA(n_components=0.99)
pca.fit(X_train)
X_train_pca = pca.transform(X_train)
X_test_pca = pca.transform(X_test)

data_pca = (X_train_pca, X_test_pca, y_train, y_test)

evaluate(model, data_pca)

0.7032967032967034

In [67]:
X_train.shape

(361, 279)

In [68]:
d = X_train_pca.shape[1]
d

63

# Redução de dimensionalidade por seleção de características

Pode melhorar em situações em que exista características redundantes ou irrelevantes.

Se não feita com cuidado resulta em overfiting e resultados enviesados.

# Tipos de seleção de características

*   Filter
*   Wrapper
*   Embedded

Todos pode ser unimodal ou multimodal.



In [84]:
import numpy as np

def varfs(data, d):
  X_train, X_test, y_train, y_test = data
  variances = X_train.var(axis=0)
  feat_idx = np.argsort(variances)[::-1]
  X_train_varfs = X_train[:, feat_idx[:d]]
  X_test_varfs = X_test[:, feat_idx[:d]]
  data_varfs = (X_train_varfs, X_test_varfs, y_train, y_test)
  return data_varfs, feat_idx[:d]

data_varfs, selected_feats = varfs(data_original, d)
print(f"Accuracy with {d} feats with highest variance\n{evaluate(model, data_varfs)}\n")
print(f"{len(selected_feats)} of {X_train.shape[1]} features\n{selected_feats}")

Accuracy with 63 feats with highest variance
0.6703296703296703

63 of 259 features
[ 10  13   9   5   2   7 248  12 258   6 247  11  56  87 238  51   8  41
 237 268 257  89  40  64  65  77 149  39  99  29 227  76  17  53 278  16
  44 178  92  63  28 188 101 228 137  52   3 125   0 267 218 187 113   4
  75  68 111 148 177 100 217  88 277]


In [70]:
from sklearn.feature_selection import VarianceThreshold
selector = VarianceThreshold(threshold=0)
selector.fit(X_train)
X_train_varth = selector.transform(X_train)
X_test_varth = selector.transform(X_test)
data_varth = (X_train_varth, X_test_varth, y_train, y_test)

print(X_train_varth.shape)
evaluate(model, data_varth)

(361, 259)


0.5824175824175825

In [71]:
from sklearn.feature_selection import SelectKBest, f_classif
X_train, X_test, y_train, y_test = data_varth
selector = SelectKBest(f_classif, k=d)
selector.fit(X_train, y_train)
X_train_kbest = selector.transform(X_train)
X_test_kbest = selector.transform(X_test)
data_kbest = (X_train_kbest, X_test_kbest, y_train, y_test)

evaluate(model, data_kbest)

0.6483516483516484

In [76]:
from sklearn.model_selection import train_test_split

def sequential_forward_selection(model, data, d):
  X_train, X_test, y_train, y_test = data
  X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2)
  best_feats = []
  remaining_feats = list(range(X_tr.shape[1]))
  for f in range(d):
    best_score = -np.inf
    best_feat = -1
    for i in remaining_feats:
      print(20*"\b", f"{f+1}/{d}: {i}/{len(remaining_feats)}", end="")
      best_feats.append(i)
      model.fit(X_tr[:, best_feats], y_tr)
      score = model.score(X_val[:, best_feats], y_val)
      if score > best_score:
        best_score = score
        best_feat = i
      best_feats = best_feats[:-1]
    best_feats.append(best_feat)
    remaining_feats.remove(best_feat)
  data_fwsel = (X_train[:, best_feats], X_test[:, best_feats], y_train, y_test)
  return data_fwsel

data_fwsel = sequential_forward_selection(model, data_original, d)
evaluate(model, data_fwsel)

 1/63: 0/279 1/63: 1/279 1/63: 2/279 1/63: 3/279 1/63: 4/279 1/63: 5/279 1/63: 6/279 1/63: 7/279 1/63: 8/279 1/63: 9/279 1/63: 10/279 1/63: 11/279 1/63: 12/279 1/63: 13/279 1/63: 14/279 1/63: 15/279 1/63: 16/279 1/63: 17/279 1/63: 18/279 1/63: 19/279 1/63: 20/279 1/63: 21/279 1/63: 22/279 1/63: 23/279 1/63: 24/279 1/63: 25/279 1/63: 26/279 1/63: 27/279 1/63: 28/279 1/63: 29/279

0.6373626373626373

In [121]:
import numpy as np

def variance_score(X, y=None):
  return X.var(axis=0)[0]

def feature_ranking(X, y=None, n_feats=None, score_function=variance_score):
  n_feats = X.shape[1] if n_feats is None else n_feats
  scores = []
  for f in range(X.shape[1]):
    score = score_function(X[:, f:f+1], y)
    scores.append(score)
  sorted_feats = np.argsort(scores)[::-1]
  X_selected = X[:, sorted_feats[:n_feats]]
  return X_selected, sorted_feats[:n_feats]

X_train_feat_rank, selected_feats = feature_ranking(X_train, n_feats=d)
X_test_feat_rank = X_test[:, selected_feats]
data_feat_rank = (X_train_feat_rank, X_test_feat_rank, y_train, y_test)
print(f"Accuracy with {d} feats with highest variance\n{evaluate(model, data_feat_rank)}\n")
print(f"{len(selected_feats)} of {X_train.shape[1]} features\n{selected_feats}")

Accuracy with 63 feats with highest variance
0.6703296703296703

63 of 259 features
[ 10  13   9   5   2   7 230  12 240   6 229  11  54  80 220  49   8  39
 219 249 239  82  38  62  63  72 136  37  92  27 209  71  17  51 258  16
  42 161  85  61  26 171  94 210 128  50   3 118   0 248 200 170 106   4
  70  65 104 135 160  93 199  81 257]


In [99]:
def mean_intra_class_distance(X, y):
  distances = []
  for c in np.unique(y):
    d = np.linalg.norm(X[y == c] - X[y == c].mean(axis=0), axis=1)
    distances.append(d.mean())
  return np.mean(distances)

mean_intra_class_distance(X_train, y_train)

np.float64(186.54623119329523)

In [100]:
def mean_inter_class_distance(x, y):
  distances = []
  for c1 in np.unique(y):
    for c2 in np.unique(y):
      X1 = x[y == c1]
      X2 = x[y == c2]
      d = np.linalg.norm(X1.mean(axis=0) - X2.mean(axis=0))
      distances.append(d)
  return np.mean(distances)

mean_inter_class_distance(X_train, y_train)

np.float64(212.25848840768904)

In [122]:
def distance_criterion(X, y):
  intra_dist = mean_intra_class_distance(X, y)
  inter_dist = mean_inter_class_distance(X, y)
  return inter_dist / intra_dist

distance_criterion(X_train, y_train)

np.float64(1.137833163660923)

In [125]:
X_train_feat_rank, selected_feats = feature_ranking(X_train, y_train,
                                                    n_feats=d,
                                                    score_function=distance_criterion)
X_test_feat_rank = X_test[:, selected_feats]
data_feat_rank = (X_train_feat_rank, X_test_feat_rank, y_train, y_test)
print(f"Accuracy with {d} feats with best distance\n{evaluate(model, data_feat_rank)}\n")
print(f"{len(selected_feats)} of {X_train.shape[1]} features\n{selected_feats}")

Accuracy with 63 feats with best distance
0.6593406593406593

63 of 259 features
[209 135 104 106 219   5  16 222 211 257  26   8  39   4  14  92 180 229
 201 163  30 127  19   7 131 165  83 239 221 153 172 218 149 253 105  13
 208  50  37  93  65 152 244   6 160  25 240 113 212 250 194   0 228  54
 179 216  96 174 232 252  61 138  24]
